# Notebook 02 — Graph Construction

## Purpose
Build a clean, filtered, attributed directed graph from the PaySim transaction 
data and save it to disk for use in all subsequent notebooks.

## Inputs
- `data/raw/PS_20174392719_1491204439457_log.csv`

## Outputs
- `data/processed/transactions_filtered.csv` — cleaned transaction dataframe
- `data/processed/transaction_graph.gpickle` — serialised NetworkX graph

## Decisions applied
- TRANSFER and CASH_OUT transactions only
- Zero-amount transactions removed
- Directed weighted graph (DiGraph)
- Edge attributes: amount, transaction type, fraud label, time step
- Node attributes: in-degree, out-degree, fraud label

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import os
import pickle

# Load raw data

df = pd.read_csv('../data/raw/PS_20174392719_1491204439457_log.csv')
print(f"Raw dataset: {df.shape}")


Raw dataset: (6362620, 11)


In [2]:
#Step1 :Filter to TRANSFER and CASH_OUT only
df_filtered = df[(df['type'] == 'TRANSFER') | (df['type'] == 'CASH_OUT')]
print(f"After type filter: {df_filtered.shape}")

#Step2 : Remove zero-amount transactions
df_filtered = df_filtered[df_filtered['amount'] != 0]
print(f"After removing zero-amount transactions: {df_filtered.shape}")

#Step3: Reset the index
df_filtered = df_filtered.reset_index(drop=True)
print(f"\nFinal filtered dataset: {df_filtered.shape}")
print(f"\nTransaction types remaining :\n{df_filtered['type'].value_counts()}")
print(f"\nFraud count: {df_filtered['isFraud'].sum()}")
print(f"Fraud percentage: {df_filtered['isFraud'].mean() * 100:.4f}%")

After type filter: (2770409, 11)
After removing zero-amount transactions: (2770393, 11)

Final filtered dataset: (2770393, 11)

Transaction types remaining :
type
CASH_OUT    2237484
TRANSFER     532909
Name: count, dtype: int64

Fraud count: 8197
Fraud percentage: 0.2959%


In [3]:
# Save filtered dataframe to processed folder
os.makedirs('../data/processed', exist_ok=True)
df_filtered.to_csv('../data/processed/filtered_transactions.csv', index=False)
print(f"Filtered dataset saved to '../data/processed/filtered_transactions.csv'")
print(f"Filtered dataset: {df_filtered.shape}")

Filtered dataset saved to '../data/processed/filtered_transactions.csv'
Filtered dataset: (2770393, 11)


In [4]:
# Build directed weighted graph 
G = nx.DiGraph()

# Add egdes more efficiently using pandas

edges = list(zip(
    df_filtered['nameOrig'],
    df_filtered['nameDest'],
    df_filtered['amount'],  
    df_filtered['type'],
    df_filtered['isFraud'],
    df_filtered['step']
))



for orig, dest, amount, trans_type,isFraud, step in edges:
    G.add_edge(orig, dest, weight=amount, is_fraud=isFraud, transaction_type=trans_type, step=step)
print(f"Graph constructed with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

Graph constructed with 3277489 nodes and 2770393 edges.


In [5]:
# Compute node level attributes
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())

# Identify fraud nodes
fraud_orginators = set(df_filtered[df_filtered['isFraud'] == 1]['nameOrig'])
fraud_recipients = set(df_filtered[df_filtered['isFraud'] == 1]['nameDest'])

# Set node attributes
for node in G.nodes():
    G.nodes[node]['in_degree'] = in_deg[node]
    G.nodes[node]['out_degree'] = out_deg[node]
    G.nodes[node]['is_fraud_originator'] = 1 if node in fraud_orginators else 0
    G.nodes[node]['is_fraud_recipient'] = 1 if node in fraud_recipients else 0
    
# veify
sample_node = list(G.nodes())[0]
print(f"\nSample node: {sample_node}:")
print(f"Attributes: {G.nodes[sample_node]}")
print(f"\nFraud originators: {len(fraud_orginators):,}")
print(f"Fraud recipients: {len(fraud_recipients):,}")


Sample node: C1305486145:
Attributes: {'in_degree': 0, 'out_degree': 1, 'is_fraud_originator': 1, 'is_fraud_recipient': 0}

Fraud originators: 8,197
Fraud recipients: 8,154


In [16]:
# Save graph using pickle
with open('../data/processed/transaction_graph.pkl', 'wb') as f:
    pickle.dump(G, f)
    
# verify file was saved


In [18]:
file_size = os.path.getsize('../data/processed/transaction_graph.pkl')
print(f"\nGraph saved to '../data/processed/transaction_graph.pkl' with size {file_size / (1024 * 1024):.2f} MB")


Graph saved to '../data/processed/transaction_graph.pkl' with size 319.93 MB


In [17]:
# Veifity the saved graph loads correclty

with open('../data/processed/transaction_graph.pkl', 'rb') as f:
    G_loaded = pickle.load(f)
    
print(f"Nodes: {G_loaded.number_of_nodes()}")
print(f"Edges: {G_loaded.number_of_edges()}")
print(f"Is directed: {G_loaded.is_directed()}")

sample_node = list(G_loaded.nodes())[0]
print(f"\nSample node: {sample_node}:")
print(f"Attributes: {G_loaded.nodes[sample_node]}")

assert G_loaded.number_of_nodes() == G.number_of_nodes()
assert G_loaded.number_of_edges() == G.number_of_edges()
print(f"\nVerification successful: Loaded graph matches the original graph.")

Nodes: 3277489
Edges: 2770393
Is directed: True

Sample node: C1305486145:
Attributes: {'in_degree': 0, 'out_degree': 1, 'is_fraud_originator': 1, 'is_fraud_recipient': 0}

Verification successful: Loaded graph matches the original graph.


## Steps taken
1. Filtered to TRANSFER and CASH_OUT transactions only (2,770,409 → 2,770,393 
   after removing 16 zero-amount transactions)
2. Built directed weighted graph — 3,277,489 nodes, 2,770,393 edges
3. Stored edge attributes: amount, transaction type, fraud label, time step
4. Stored node attributes: in-degree, out-degree, fraud originator label, 
   fraud receiver label
5. Saved and verified graph to disk at 319MB